# Projet #3 — Prédiction du Churn Client
## Notebook 1 — Analyse, qualité et préparation des données

Ce notebook répond au **premier livrable** du brief : analyser la qualité du jeu de données, produire une analyse statistique descriptive avec visualisations, puis définir une procédure de nettoyage pour rendre les données exploitables par un processus d'apprentissage.

**Objectifs :**
- comprendre la structure du dataset Telco Customer Churn ;
- identifier valeurs manquantes, incohérences et doublons ;
- comparer les profils des clients qui restent et de ceux qui churnent ;
- nettoyer `TotalCharges` et préparer la cible ;
- produire un jeu de données propre pour le notebook de modélisation.

> Chaque ligne représente un client. La cible `Churn` indique si le client a quitté l'entreprise le mois précédent.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

RAW_CANDIDATES = list(Path("../data/raw").glob("WA_Fn-UseC_-Telco-Customer-Churn*.csv"))
if not RAW_CANDIDATES:
    raise FileNotFoundError(
        "Placez le CSV Kaggle dans data/raw/ sous un nom commençant par "
        "'WA_Fn-UseC_-Telco-Customer-Churn'."
    )

DATA_PATH = RAW_CANDIDATES[0]
df = pd.read_csv(DATA_PATH)
print(f"Fichier chargé : {DATA_PATH}")
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()

## 1. Qualité des données

On vérifie la structure, les types, les doublons, les valeurs manquantes et les valeurs atypiques. Une attention particulière est portée à `TotalCharges`, qui est importée comme texte alors qu'elle représente un montant numérique.


In [ ]:
quality = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "valeurs_manquantes": df.isna().sum(),
    "nb_uniques": df.nunique(dropna=False)
})
quality

In [ ]:
print("Doublons exacts :", df.duplicated().sum())
print("Identifiants clients dupliqués :", df["customerID"].duplicated().sum())

blank_total = df["TotalCharges"].astype(str).str.strip().eq("")
print("TotalCharges vides/non renseignés :", int(blank_total.sum()))

print("\nRépartition de la cible :")
display(df["Churn"].value_counts())
display((df["Churn"].value_counts(normalize=True) * 100).round(2).rename("pourcentage"))

### Interprétation qualité

- `customerID` est un identifiant technique : il ne doit pas être utilisé comme variable prédictive.
- `TotalCharges` contient quelques chaînes vides. Elles deviennent des `NaN` lors de la conversion numérique.
- Les lignes concernées correspondent typiquement à des clients avec une ancienneté très faible / nulle ; on choisit ici une **imputation médiane** dans le pipeline ML afin de conserver ces clients.
- La cible est déséquilibrée : le churn représente environ un quart des clients. L'accuracy seule serait donc insuffisante ; on suivra aussi précision, rappel, F1 et ROC-AUC.


In [ ]:
# Conversion contrôlée de TotalCharges
df_clean = df.copy()
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

print("NaN après conversion de TotalCharges :", df_clean["TotalCharges"].isna().sum())
display(df_clean.loc[df_clean["TotalCharges"].isna(),
                     ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head(15))

## 2. Statistiques descriptives


In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df_clean[numeric_cols].describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df_clean, x=col, hue="Churn", kde=True, element="step", ax=ax)
    ax.set_title(f"Distribution de {col} selon le churn")
plt.tight_layout()
plt.show()

### Lecture

Ces distributions permettent d'observer si les clients qui churnent ont une ancienneté ou des niveaux de facturation différents de ceux qui restent. L'objectif n'est pas de conclure à une causalité, mais d'identifier des variables potentiellement prédictives.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    rate = (df_clean.groupby(col)["Churn"]
            .apply(lambda s: (s == "Yes").mean())
            .sort_values(ascending=False)
            .mul(100)
            .rename("churn_rate")
            .reset_index())
    sns.barplot(data=rate, x=col, y="churn_rate", ax=ax, color="steelblue")
    ax.set_title(f"Taux de churn par {col}")
    ax.set_ylabel("Churn (%)")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
churn_numeric = df_clean.assign(ChurnBinary=(df_clean["Churn"] == "Yes").astype(int))
corr = churn_numeric[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "ChurnBinary"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Corrélations entre variables numériques et churn")
plt.tight_layout()
plt.show()
corr["ChurnBinary"].sort_values(ascending=False)

## 3. Analyse ciblée des caractéristiques

On examine quelques variables métier importantes : type de contrat, service Internet, support technique, sécurité en ligne et ancienneté. Les écarts observés guideront ensuite l'interprétation des modèles.


In [ ]:
categorical_focus = ["Contract", "InternetService", "TechSupport", "OnlineSecurity", "PaperlessBilling"]

for col in categorical_focus:
    table = pd.crosstab(df_clean[col], df_clean["Churn"], normalize="index").mul(100).round(1)
    print(f"\n--- {col} ---")
    display(table)

In [ ]:
# Churn selon des classes d'ancienneté
bins = [-1, 6, 12, 24, 48, 72]
labels = ["0-6 mois", "7-12 mois", "13-24 mois", "25-48 mois", "49-72 mois"]
df_clean["tenure_group"] = pd.cut(df_clean["tenure"], bins=bins, labels=labels)

tenure_churn = (df_clean.groupby("tenure_group", observed=False)["Churn"]
                .apply(lambda s: (s == "Yes").mean())
                .mul(100)
                .rename("churn_rate")
                .reset_index())

plt.figure(figsize=(9, 4))
sns.barplot(data=tenure_churn, x="tenure_group", y="churn_rate", color="steelblue")
plt.ylabel("Taux de churn (%)")
plt.xlabel("Ancienneté")
plt.title("Taux de churn selon l'ancienneté client")
plt.tight_layout()
plt.show()

## 4. Procédure de nettoyage retenue

1. Conserver une copie du dataset brut.
2. Convertir `TotalCharges` en numérique avec `errors="coerce"`.
3. Retirer `customerID` des variables d'apprentissage.
4. Transformer `Churn` en cible binaire (`Yes` → 1, `No` → 0).
5. Ne pas imputer/encoder avant la séparation train/test : ces opérations seront placées dans un `Pipeline` scikit-learn afin d'éviter les fuites de données.
6. Imputer les variables numériques par médiane.
7. Encoder les catégories avec `OneHotEncoder(handle_unknown="ignore")`.
8. Standardiser les variables numériques pour la régression logistique.

Le groupe d'ancienneté créé ci-dessus sert uniquement à l'EDA et n'est pas conservé comme variable d'entrée, car `tenure` contient déjà l'information originale.


In [ ]:
# Préparation d'un fichier propre pour le notebook 2
prepared = df_clean.drop(columns=["tenure_group"]).copy()
prepared["Churn"] = prepared["Churn"].map({"No": 0, "Yes": 1}).astype(int)

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "telco_churn_clean.csv"
prepared.to_csv(OUTPUT_PATH, index=False)

print(f"Jeu nettoyé enregistré : {OUTPUT_PATH}")
print("Dimensions :", prepared.shape)
print("NaN TotalCharges conservés pour imputation dans le pipeline :", prepared["TotalCharges"].isna().sum())
prepared.head()

## 5. Conclusions de l'EDA

Le dataset est globalement exploitable mais nécessite une correction de type pour `TotalCharges`. La cible est déséquilibrée, ce qui impose d'évaluer les modèles avec plusieurs métriques.

Les analyses descriptives suggèrent que l'ancienneté, le type de contrat, certains services Internet et le niveau de facturation sont liés au churn. Ces observations seront vérifiées dans le notebook suivant à l'aide de modèles supervisés et de l'importance des caractéristiques.
